# 경기도 카드 소비 데이터 매출 예측 모델 학습 (전체 데이터)

## 사전 준비
1. 캐글 데이터셋에 CSV 파일들 업로드 후 Input 탭에서 연결
2. Session Options → Accelerator: GPU T4 x2
3. Run All 실행 후 Output 탭에서 모델 파일 다운로드

### 메모리 최적화 전략
- OHE 대신 LightGBM 네이티브 카테고리 처리 (OHE 행렬 불필요)
- dtype 최적화 (int8, int32, category)
- 파일별 로드 후 즉시 gc

In [ ]:
import os, glob, time, gc
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 캐글 Input 탭에서 연결한 데이터셋 경로 확인 후 수정
DATA_DIR   = "/kaggle/input/gyeonggi-card-data"
OUTPUT_DIR = "/kaggle/working"

os.makedirs(f"{OUTPUT_DIR}/model",    exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/encoders", exist_ok=True)

files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"파일 수: {len(files)}개")
for f in files[:5]:
    print(f"  {os.path.basename(f)}")
print("  ...")

In [ ]:
# =========================================================
# 설정
# =========================================================
# LightGBM 네이티브 카테고리 사용 → OHE 없음
CAT_COLS  = ["sex", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2"]
NUM_COLS  = ["age", "day", "hour", "month", "cnt"]
ALL_FEATS = NUM_COLS + CAT_COLS

REMOVE_OUTLIER = True

DTYPES = {
    "admi_cty_no":     "int32",
    "hour":            "int8",
    "sex":             "category",
    "age":             "int8",
    "day":             "int8",
    "amt":             "int64",
    "cnt":             "int32",
    "card_tpbuz_nm_1": "category",
    "card_tpbuz_nm_2": "category",
}

In [ ]:
# =========================================================
# 1) 전체 데이터 로드
# =========================================================
dfs = []
t0 = time.time()

for i, f in enumerate(files, 1):
    city = os.path.basename(f).split("_")[-1].replace(".csv", "")
    df = pd.read_csv(
        f, encoding="utf-8-sig",
        usecols=["ta_ymd", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2",
                 "hour", "sex", "age", "day", "amt", "cnt"],
        dtype=DTYPES
    )
    df["month"] = df["ta_ymd"].astype(str).str[4:6].astype("int8")
    df["admi_cty_no"] = df["admi_cty_no"].astype("category")
    df["log_amt"] = np.log1p(df["amt"]).astype("float32")

    if REMOVE_OUTLIER:
        upper = df["amt"].quantile(0.99)
        df = df[df["amt"] <= upper]

    dfs.append(df[ALL_FEATS + ["log_amt"]].dropna())
    if i % 10 == 0 or i == len(files):
        print(f"[{i}/{len(files)}] {city} 완료  ({time.time()-t0:.0f}s)")

df_all = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()

# category 컬럼 재설정 (concat 후 dtype 유지)
for col in CAT_COLS:
    df_all[col] = df_all[col].astype("category")

print(f"\n전체: {len(df_all):,}행")
print(f"메모리: {df_all.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# =========================================================
# 2) Label Encoding (LightGBM 카테고리용)
#    OHE 없음 → 메모리 절약
# =========================================================
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_all[col] = le.fit_transform(df_all[col].astype(str)).astype("int16")
    label_encoders[col] = le

feature_columns = ALL_FEATS
X = df_all[feature_columns]
y = df_all["log_amt"]
del df_all; gc.collect()

print(f"피처: {feature_columns}")
print(f"메모리: {X.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# =========================================================
# 3) 학습 / 검증 분할
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
del X, y; gc.collect()

print(f"학습: {len(X_train):,}행 / 검증: {len(X_test):,}행")

In [ ]:
# =========================================================
# 4) LightGBM 학습 (네이티브 카테고리)
# =========================================================
t0 = time.time()

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=CAT_COLS,   # OHE 없이 LightGBM이 직접 처리
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)

print(f"\n학습 완료: {time.time()-t0:.1f}초")

In [ ]:
# =========================================================
# 5) 평가
# =========================================================
pred = model.predict(X_test)
y_real    = np.expm1(y_test)
pred_real = np.maximum(np.expm1(pred), 0)

r2   = r2_score(y_real, pred_real)
rmse = np.sqrt(mean_squared_error(y_real, pred_real))
mae  = mean_absolute_error(y_real, pred_real)

print(f"R²   = {r2:.4f}")
print(f"RMSE = {rmse:,.0f}")
print(f"MAE  = {mae:,.0f}")

In [ ]:
# =========================================================
# 6) 저장
# =========================================================
joblib.dump(model,          f"{OUTPUT_DIR}/model/sales_predict_model.pkl")
joblib.dump(label_encoders, f"{OUTPUT_DIR}/encoders/label_encoders.pkl")
joblib.dump(feature_columns,f"{OUTPUT_DIR}/encoders/feature_columns.pkl")
joblib.dump({
    "model_name":      "LightGBM",
    "R2":              r2,
    "RMSE":            rmse,
    "MAE":             mae,
    "total_rows":      len(X_train) + len(X_test),
    "features":        feature_columns,
    "cat_cols":        CAT_COLS,
    "use_log_target":  True,
    "remove_outliers": REMOVE_OUTLIER,
    "n_files":         len(files),
    "data_start":      pd.Series([os.path.basename(f) for f in files]).str.extract(r"_(\d{6})_")[0].dropna().min(),
    "data_end":        pd.Series([os.path.basename(f) for f in files]).str.extract(r"_(\d{6})_")[0].dropna().max(),
    "encoding":        "label",   # OHE 아님
}, f"{OUTPUT_DIR}/model/model_info.pkl")

print("저장 완료!")
print("Output 탭에서 다운로드:")
print("  model/sales_predict_model.pkl")
print("  model/model_info.pkl")
print("  encoders/label_encoders.pkl")
print("  encoders/feature_columns.pkl")